# Phase 5 — Centralized SAC Production Training Runner
**Project:** Physiological Coordinated Control of Biventricular Artificial Hearts Using Deep Reinforcement Learning  
**Target Model:** Joint Centralized Soft Actor-Critic (9-dim observation, 2-dim continuous speed action)

---
### Multi-Notebook Kaggle Setup:
- **Session Type:** Notebook
- **Accelerator:** **None (CPU-only)** — *Allocates 4 vCPUs for parallel ODE solving without consuming GPU quotas!*
- **Multi-Seed Strategy:** Duplicate this notebook 5 times on Account 1, setting `SEED = 42`, `123`, `456`, `789`, `1024` in Block 0.

### Block 0: Seed & Step Budget Configuration
**Change `SEED` for each notebook instance** (e.g., 42, 123, 456, 789, 1024).

In [ ]:
# ==============================================================================
# RUNNER CONFIGURATION — EDIT SEED FOR EACH OF YOUR 5 PARALLEL NOTEBOOKS
# ==============================================================================
SEED = 42          # Options: 42, 123, 456, 789, 1024
TOTAL_STEPS = 500000
N_WORKERS = 4      # Auto-detects 4 vCPUs on Kaggle

print(f"Configured Centralized SAC Runner: Seed={SEED}, Target Steps={TOTAL_STEPS:,}, Workers={N_WORKERS}")

### Block 1: Environment Setup & File Extraction
Locates and copies project files from the attached Kaggle dataset to `/kaggle/working`.

In [ ]:
# 1. Install dependencies
!pip install gymnasium stable-baselines3 scipy torch tensorboard

# 2. Locate and copy/extract dataset files
import os, glob, shutil, zipfile, multiprocessing

zip_candidates = (
    glob.glob('/kaggle/input/**/kaggle_training_test.zip', recursive=True) +
    glob.glob('/kaggle/input/**/kaggle_bivad_phase5.zip', recursive=True) +
    glob.glob('kaggle_training_test.zip') +
    glob.glob('kaggle_bivad_phase5.zip')
)
bivad_env_matches = glob.glob('/kaggle/input/**/bivad_env.py', recursive=True)

if zip_candidates:
    target_zip = zip_candidates[0]
    print(f"Found archive: {target_zip}. Unzipping to /kaggle/working...")
    with zipfile.ZipFile(target_zip, 'r') as z:
        z.extractall('.')
elif bivad_env_matches:
    source_dir = os.path.dirname(bivad_env_matches[0])
    print(f"Found extracted dataset at: {source_dir}. Copying files to /kaggle/working...")
    for item in os.listdir(source_dir):
        s = os.path.join(source_dir, item)
        d = os.path.join('.', item)
        if os.path.isdir(s):
            if os.path.exists(d): shutil.rmtree(d)
            shutil.copytree(s, d)
        else:
            shutil.copy2(s, d)
else:
    print("[NOTE] Assuming files are already in current directory.")

# 3. Verify core components
required_items = ['simulator', 'controllers', 'bivad_env.py', 'reward.py', 'train_centralized.py', 'post_training_evaluation.py', 'expert_data_pi.npz']
all_ok = True
for item in required_items:
    exists = os.path.exists(item)
    print(f"{item:25s}: {'[FOUND]' if exists else '[MISSING]'}")
    if not exists: all_ok = False

print(f"\nDetected CPU Cores: {multiprocessing.cpu_count()}")
if all_ok:
    print(">>> SUCCESS: Environment ready! Proceed to training.")
else:
    print(">>> ERROR: Missing files detected. Please verify dataset attachment.")

### Block 2: Behavioral Cloning Warm-Start (Pretrain Centralized Actor)
Pretrains the SAC Actor network on 400,000 transitions from `expert_data_pi.npz` and runs a 200s Rest sanity check.

In [ ]:
!python train_centralized.py --mode bc --seed {SEED} --expert_data expert_data_pi.npz --device cpu

### Block 3: Full Production SAC Training (500,000 Timesteps)
- Checkpoints every 100,000 steps with running `VecNormalize` statistics.
- Logs multi-objective reward subcomponents (`r_track`, `r_balance`, `r_safety`, `r_smooth`, `r_map`) into TensorBoard.
- Automatically executes 9-scenario clinical validation post-training.
- Generates publication figures (`plots/`) and performance tables (`tables/`).
- **Auto-Resumable:** If Kaggle session times out, re-running this cell automatically picks up from the latest checkpoint!

In [ ]:
!python train_centralized.py --mode full --seed {SEED} --full_steps {TOTAL_STEPS} --n_envs {N_WORKERS} --device cpu

### Block 4: Package Checkpoints & Download Seed Results
Compresses checkpoints and TensorBoard events for this specific seed into a single zip file.

In [ ]:
!zip -r centralized_sac_seed_{SEED}.zip checkpoints/seed_{SEED}/ tb_centralized/seed_{SEED}/
print(f"\n>>> [READY TO DOWNLOAD] 'centralized_sac_seed_{SEED}.zip' is ready!\n"\
      f"    Includes: Trained models, normalizers, 9-scenario performance tables, publication figures, and TensorBoard logs.\n"\
      f"    Download it directly from the Kaggle Output panel on the right.")